# Inter-rater reliability Script


Takes in a folder of csv files from ELAN, with file names {video_name}_ {initials}. Standardizes behavior names to "allogrooming", "allolicking", and "self licking". Outputs corrleational matrix between each person (for all videos together) and ICC score.

**folder of csv data must be in same directory as this jupyter notebook, change folder path if needed**

In [1]:
#Import necessary libraries
#you may need to install some of these packages if you haven't already
import pandas as pd
import numpy as np
import pingouin as pg
from sklearn.metrics import cohen_kappa_score
import os
import re
import seaborn as sns
import matplotlib.pyplot as plt

/Users/nikkikeisler1/miniconda3/envs/simba/lib/python3.6/site-packages/outdated/utils.py:18: OutdatedPackageWarning: The package pingouin is out of date. Your version is 0.4.0, the latest is 0.5.5.
Set the environment variable OUTDATED_IGNORE=1 to disable these warnings.
  **kwargs


In [2]:
# assuming all data is in same folder as this file, with folder name 'csv'
folder_path = r"/Volumes/Lab/Reliability/socialInteractionBehaviors/data/202509_allogrooming_allolicking/csv"
#Input initials of the user to compare all other scores against
initial_1 = "NK" 

In [3]:
#Create function to standardize behavior names
def standardize(x):
    '''
    takes a column of text, and standardizes it to all lowercase, no spaces, no punctuation/non-alphanumeric characters
    '''
    # Remove all non-alphanumeric characters (punctuation, spaces, etc.)
    label_cleaned = re.sub(r'[^a-zA-Z]', '', x)
    # Convert to lowercase
    label_standardized = label_cleaned.lower()
    
#    if 'self' in label_standardized:
#        label_standardized = 'selflicking'
#    elif 'groom' in label_standardized:
#        label_standardized = 'allogrooming'
#    else:
#        label_standardized = 'allolicking'
#    
    return label_standardized

In [7]:
#Create data frames from each csv and initiate pairwise ICC comparisons

# dictionary to hold all the data that will be turned into final df
all_video_data = {}
large_df = pd.DataFrame()
all_initials = set()

# loop through each file 
for filename in os.listdir(folder_path):
    
    if filename.endswith('.csv'):  # Check if the file is a CSV
        file_path = os.path.join(folder_path, filename)
        
        # temp dictionary that'll be added to all_video_data at the end of the loop
        curr_vid_data = {}
        
        # extract initials and file name for labeling purposes
        # Split filename by underscore and take the last part (before .csv) as initials
        filename_parts = filename.replace('.csv', '').split('_')
        initials = filename_parts[-1]  # This will handle both 2 and 3 letter initials
        video_name = '_'.join(filename_parts[:-1])  # Everything except the last part is the video name

        all_initials.add(initials)

    
   
        # turn current csv file into a data frame
        df = pd.read_csv(file_path, header = None)
        
        # standardize the labeling to all lowercase and no spaces/punctuation
        df[0] = df[0].apply(standardize)
        
        # get the total duration of each behavior
        # 0 = behavior name
        # 1 = NaN column
        # 2 = start time
        # 3 = end time
        # 4 = duration
        # 5 = NaN column
        
        d = dict(df.groupby(0).sum()[4])
        
        # rename the keys using initial and behavior name ex) allogrooming_KT
        for key, value in d.items():
            curr_vid_data[f'{key}_{initials}'] = value
        
        
        # first time this video appears,  not yet in large dict (creates new row for the video)
        if video_name not in all_video_data:
            all_video_data[video_name] = curr_vid_data  
            
        # video already in the dictonary, just append the data to existing row
        else:
            all_video_data[video_name].update(curr_vid_data)      
        

# Create a dataframe per behavior with column = initials, rows = video and values = total duration of behavior
reliability_df = pd.DataFrame(all_video_data).T.sort_index().fillna(0)

# Create all pairwise comparisons of unique values in a list
initials_list = list(all_initials)
pairwise_comparisons = [(g1, g2) for i, g1 in enumerate(initials_list) for g2 in initials_list[i+1:]]
output = {}
behaviors = ['allogrooming', 'allolicking', 'selflicking']

for behavior in behaviors:
#     print(f'{behavior}')
    col = [i for i in reliability_df.columns if behavior in i]
    behavior_df = reliability_df[col]
    behavior_df.columns = behavior_df.columns.str.split('_').str[-1]  # Extract initials from column names
#     raters = df_wide.columns
    
    icc_results = {}
    
    for rater in behavior_df.columns:
        if rater != initial_1:
            temp_df = behavior_df[[initial_1, rater]].reset_index()  # Ensure video names are included
            temp_df = temp_df.melt(id_vars="index", var_name="Rater", value_name="Score")

            icc_result = pg.intraclass_corr(data=temp_df, targets='index', raters='Rater', ratings='Score')
            icc_value = icc_result[icc_result['Type'] == 'ICC2']['ICC'].values[0]
            icc_results[rater] = icc_value  # Store the ICC value
    
    output[behavior] = icc_results

    # Convert to DataFrame
    icc_df = pd.DataFrame.from_dict(icc_results, orient='index', columns=[f'ICC with {initial_1} for {behavior}']).reset_index()

    # Print ICC values
    print(icc_df)

   index  ICC with NK for allogrooming
0     AN                      0.991274
1     DS                      0.828308
2     BF                      0.957483
3     AZ                      0.800500
4    PRB                      0.972402
5     AB                      0.732104
6     MW                      0.948557
7     KT                      0.804420
8     KC                      0.566382
9     MR                      0.776782
10    OE                      0.977743
11   KLT                      0.917445
   index  ICC with NK for allolicking
0     AN                     0.997791
1     DS                     0.993969
2     BF                     0.985166
3     MA                     0.914448
4     AZ                     0.997832
5    PRB                     0.994527
6     AB                     0.997274
7     MW                     0.998965
8     KT                     0.991724
9     KC                     0.993269
10    MR                     0.998108
11    OE                     0.997961

In [ ]:
#Plot ICC result heat map with user-selected active lab members

# Display all available initials
print("Available lab members (initials):")
all_available_initials = set()
for behavior_data in output.values():
    all_available_initials.update(behavior_data.keys())
all_available_initials.add(initial_1)  # Add the reference initial
sorted_initials = sorted(list(all_available_initials))

for i, initial in enumerate(sorted_initials, 1):
    print(f"{i}. {initial}")

print(f"\nReference initial (comparison baseline): {initial_1}")

# EDIT THIS LIST: Add the initials of active lab members you want to include in the plot
# Leave empty list [] to include all available initials
selected_initials = ["AN", "KC", "KLT", "MA", "MW", "NK", "PRB"]  # Edit this list as needed

# Validate that all selected initials exist in the data
if selected_initials:
    invalid_initials = [initial for initial in selected_initials if initial not in sorted_initials]
    if invalid_initials:
        print(f"Warning: The following initials were not found in the data: {invalid_initials}")
        selected_initials = [initial for initial in selected_initials if initial in sorted_initials]
else:
    # If empty list, use all available initials
    selected_initials = sorted_initials

print(f"Plotting ICC scores for: {', '.join(selected_initials)}")

# Filter the output data to include only selected initials
filtered_output = {}
for behavior, rater_data in output.items():
    filtered_output[behavior] = {rater: icc for rater, icc in rater_data.items() 
                               if rater in selected_initials}

# Create DataFrame and plot
r = pd.DataFrame(filtered_output)

# Only plot if there's data to show
if not r.empty:
    print(f"\nICC Score Comparison against {initial_1}")
    plt.figure(figsize=(10, 5))
    sns.heatmap(r, annot=True, cmap="coolwarm", linewidths=0.5, fmt=".3f")
    plt.yticks(rotation=0)
    plt.title(f'Inter-rater Reliability (ICC) - Comparison against {initial_1}')
    plt.tight_layout()
    plt.show()
else:
    print("No data available for the selected lab members.")

Available lab members (initials):
1. AB
2. AN
3. AZ
4. BF
5. DS
6. KC
7. KLT
8. KT
9. MA
10. MR
11. MW
12. NK
13. OE
14. PRB

Reference initial (comparison baseline): NK

Enter the initials of active lab members to include in the plot (separate with commas):
Example: NK,ABC,XYZ
